### ЗАДАЧА: Триаж обращений службы поддержки

Команда поддержки получает пакет строк с обращениями от разных сервисов.
Нужно обработать их так, чтобы:
- корректные обращения попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, в каких каналах остались не подтверждённые обращения,
- а также какова средняя длительность обработки по уровням приоритета.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестный уровень приоритета или канал,
а часть передаёт неправильный флаг подтверждения.
        


In [32]:
# incident_id|service|severity|duration_min|channel|acknowledged
rows = [
    'INC-100|checkout|critical|12|pager|yes',
    'INC-101|search|high|7|slack|no',
    'INC-102|billing|medium|zero|email|yes',
    'INC-103|video|critical|-3|pager|no',
    'INC-104|feed|warning|5|slack|yes',
    'INC-105|auth|low|2|sms|no',
    'INC-106|cdn|high|4|email|maybe',
    'INC-107|ml|medium|9|slack|no',
]


class IncidentProcessingError(Exception):
    pass


class IncidentFormatError(IncidentProcessingError):
    pass


class SeverityError(IncidentProcessingError):
    pass


class DurationError(IncidentProcessingError):
    pass


class ChannelError(IncidentProcessingError):
    pass


class AcknowledgedFlagError(IncidentProcessingError):
    pass


def parse_incident(row):
    # TODO: split строку по '|'
    # TODO: убрать лишние пробелы у частей через strip()
    parts = row.strip().split("|")
    # TODO: ожидать 6 частей: incident_id, service, severity, duration_raw, channel, acknowledged_raw
    incident_id, service, severity, duration_raw, channel, acknowledged_raw = parts
    # TODO: если частей не 6 -> raise IncidentFormatError(...)
    if len(parts) != 6:
        raise IncidentFormatError("Строка должна состоять из 6ти элементов.")
    # TODO: duration_raw преобразовать в float
    try:
        duration_raw = float(duration_raw)
    # TODO: при ошибке преобразования использовать raise DurationError(...) from exc
    except ValueError as e:
        raise DurationError("Продолжительность должна быть числом") from e
    # TODO: проверить, что duration > 0
    if duration_raw < 0:
        raise DurationError("Продолжительность не можеть быть отрицательной")
    # TODO: проверить severity в {'low', 'medium', 'high', 'critical'}
    allowed_severity = {'low', 'medium', 'high', 'critical'}
    if severity not in allowed_severity:
        raise SeverityError("Неправильная серьезность ошибки")
    # TODO: проверить channel в {'email', 'slack', 'pager'}
    allowed_channel = {'email', 'slack', 'pager'}
    if channel not in allowed_channel:
        raise ChannelError("Неправильный канал")
    # TODO: проверить acknowledged_raw в {'yes', 'no'}
    allowed_acknowledged = {'yes', 'no'}
    if acknowledged_raw not in allowed_acknowledged:
        raise AcknowledgedFlagError("Неправильный ответ")
    # TODO: превратить acknowledged_raw в bool
    if acknowledged_raw == "yes":
        acknowledged_raw = True
    else:
        acknowledged_raw = False
    # TODO: вернуть словарь с разобранными полями
    return {
        "incident_id": incident_id,
        "service": service,
        "severity": severity,
        "duration_raw": duration_raw,
        "channel": channel,
        "acknowledged_raw": acknowledged_raw
    }


def process_batch(rows):
    # TODO: создать списки incidents и errors
    incidents = []
    errors = []
    # TODO: пройтись по rows циклом
    for row in rows:
    # TODO: внутри try вызвать parse_incident(row)
        try:
    # TODO: валидный incident добавить в incidents
            incidents.append(parse_incident(row))
    # TODO: IncidentProcessingError сохранить в errors как (row, error_type, message)
        except IncidentProcessingError as e:
            errors.append((row, type(e).__name__, str(e)))
    # TODO: вернуть (incidents, errors)
    return incidents, errors


# TODO: вызвать process_batch(rows)
incidents, errors = process_batch(rows)
# TODO: вывести количество валидных инцидентов и количество ошибок
print(f"Число валидных инцидентов: {len(incidents)} шт.")
print(f"Число ошибок: {len(errors)} шт.")
# TODO: собрать error_counts: dict[str, int]
errors_by_type = {}
for name, error, message in errors:
    errors_by_type[error] = errors_by_type.get(error, 0) + 1
print("Ошибки по типам:")
for err, count in errors_by_type.items():
    print(f"- ошибка '{err}' встречается {count} раз.")
# TODO: собрать unacked_by_channel: dict[str, list[str]] только для acknowledged == False
unacked_by_channel = {}
for incident in incidents:
    if incident["acknowledged_raw"] == False:
        unacked_by_channel.setdefault(incident["acknowledged_raw"], [])
        unacked_by_channel[incident["acknowledged_raw"]] = [*unacked_by_channel[incident["acknowledged_raw"]], incident]
print("Неподтвержденный запрос имееют следующие строки:")
for bool, items in unacked_by_channel.items():
    for item in items:
        print(f"- {item}")
# TODO: собрать average_duration_by_severity только по валидным строкам
average_duration = 0
for incident in incidents:
    average_duration += incident["duration_raw"]
average_duration = average_duration / len(incidents)
print(f"Среднее значение инцидента по валидным строкам = {average_duration} сек.")
# TODO: найти longest_incident среди валидных инцидентов по duration_min
longest_incident = {}
for incident in incidents:
    longest_incident[incident["incident_id"]] = longest_incident.get(incident["incident_id"], incident["duration_raw"])
max_id = None
max_duration = 0
for id, count in longest_incident.items():
    if count > max_duration:
        max_id = id
        max_duration = count
print(f"Самый длинный инцидент под id '{max_id}' длился {max_duration} сек.")
# TODO: красиво вывести получившиеся структуры

Число валидных инцидентов: 3 шт.
Число ошибок: 5 шт.
Ошибки по типам:
- ошибка 'DurationError' встречается 2 раз.
- ошибка 'SeverityError' встречается 1 раз.
- ошибка 'ChannelError' встречается 1 раз.
- ошибка 'AcknowledgedFlagError' встречается 1 раз.
Неподтвержденный запрос имееют следующие строки:
- {'incident_id': 'INC-101', 'service': 'search', 'severity': 'high', 'duration_raw': 7.0, 'channel': 'slack', 'acknowledged_raw': False}
- {'incident_id': 'INC-107', 'service': 'ml', 'severity': 'medium', 'duration_raw': 9.0, 'channel': 'slack', 'acknowledged_raw': False}
Среднее значение инцидента по валидным строкам = 9.333333333333334 сек.
Самый длинный инцидент под id 'INC-100' длился 12.0 сек.
